# Exploration des données brutes OpenAgenda

Ce notebook permet d'explorer les données brutes collectées depuis l'API OpenAgenda avant leur preprocessing. L'objectif est de vérifier leur structure et leur qualité, de mesurer les éventuels doublons et d'observer des événements réels pour déterminer ensuite quels champs seront utiles au chatbot et au système RAG. Aucune transformation ni suppression de données n'est réalisée.

## 1. Chargement des données

In [21]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

# Retrouver la racine depuis le dossier du notebook ou la racine du projet.
current_directory = Path.cwd().resolve()
project_root = next(
    (directory for directory in (current_directory, *current_directory.parents)
     if (directory / 'App' / 'Data' / 'raw').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Dossier App/Data/raw introuvable : lancez le notebook depuis ce projet.')

raw_directory = project_root / 'App' / 'Data' / 'raw'
json_files = list(raw_directory.glob('openagenda_*.json'))
if not json_files:
    raise FileNotFoundError(f'Aucun fichier openagenda_*.json dans {raw_directory}.')

# La date de modification détermine le fichier le plus récent.
input_path = max(json_files, key=lambda path: (path.stat().st_mtime_ns, path.name))
with input_path.open(encoding='utf-8') as input_file:
    payload = json.load(input_file)

if not isinstance(payload, dict):
    raise ValueError('Le JSON doit contenir un objet avec une liste events.')
events = payload.get('events')
if not isinstance(events, list) or any(not isinstance(event, dict) for event in events):
    raise ValueError('Le champ events doit être une liste d’objets JSON.')

print(f'Fichier utilisé : {input_path.name}')
print(f'Événements chargés : {len(events):,}')

Fichier utilisé : openagenda_20261007T123746834561Z.json
Événements chargés : 7,222


## 2. Vérification des doublons

Les doublons sont les occurrences supplémentaires d'un même UID. Les UID absents ou invalides sont comptés séparément et exclus de ce calcul. Aucun événement n'est supprimé.

In [22]:
uids = []
missing_uid_count = 0
invalid_uid_count = 0
for event in events:
    uid = event.get('uid')
    if uid is None or (isinstance(uid, str) and not uid.strip()):
        missing_uid_count += 1
    elif type(uid) not in (str, int):
        invalid_uid_count += 1
    else:
        # Comparer 123 et '123' comme le même identifiant, sans modifier les événements.
        uids.append(str(uid))

uid_counts = Counter(uids)
duplicate_count = len(uids) - len(uid_counts)
print(f'Événements bruts : {len(events):,}')
print(f'Événements sans UID : {missing_uid_count:,}')
print(f'Événements avec un type d’UID invalide : {invalid_uid_count:,}')
print(f'UID uniques parmi les événements avec UID exploitable : {len(uid_counts):,}')
print(f'Doublons (occurrences supplémentaires) : {duplicate_count:,}')

Événements bruts : 7,222
Événements sans UID : 0
Événements avec un type d’UID invalide : 0
UID uniques parmi les événements avec UID exploitable : 3,915
Doublons (occurrences supplémentaires) : 3,307


## 3. Exploration des champs

Cette liste rassemble les clés de premier niveau rencontrées dans tous les événements. Les structures imbriquées seront visibles dans les exemples suivants.

In [23]:
fields = sorted({field for event in events for field in event})
print(f'Nombre de champs rencontrés : {len(fields)}')
for field in fields:
    print(f'- {field}')

Nombre de champs rencontrés : 60
- accessibility
- addMethod
- age
- attendanceMode
- billetterie
- canonicalUrl
- categories-pour-wwwjjulien
- categories-qds
- categories-wwwaugustins
- categories-wwwdupuy
- categories-wwwgce
- categories-wwwhttpswwwbibliothequetoulousefr-champ-obligatoire
- categories-wwwhttpswwwtoulouse-metropolefrmissionsdeveloppement-economiqueeconomie-sociale-et-solidaire
- categories-wwwjacobins
- categories-wwwmsr
- categories-wwwmuseumchamp-obligatoire
- conditions
- country
- createdAt
- creatorUid
- dateRange
- description
- draft
- entreelibre
- evenement-ponctuel
- extIds
- featured
- firstTiming
- image
- imageCredits
- keywords
- lastTiming
- links
- location
- longDescription
- motive
- nextTiming
- onlineAccessLink
- organisateur
- originAgenda
- ownerUid
- participation
- private
- public
- registration
- rights
- slug
- sourceAgendas
- state
- status
- thematiques-metropolitaines
- timezone
- timings
- title
- type-monuments
- types-de-concerts-wwwon

## 4. Trois événements au hasard

Le tirage utilise `random_state=42` pour retrouver les mêmes exemples tant que le fichier et l'ordre des événements restent identiques. S'il y a moins de trois événements, tous sont affichés.

In [24]:
# ---------- Sélection de 3 événements aléatoires ----------
if not events:
    raise ValueError("Aucun événement disponible pour les exemples.")

sample_positions = (
    pd.Series(range(len(events)), dtype="int64")
    .sample(n=min(3, len(events)), random_state=42)
    .tolist()
)

print(f"Positions sélectionnées : {sample_positions}")

Positions sélectionnées : [6282, 786, 5091]


In [25]:
# ---------- Exemple 1 ----------
events[sample_positions[0]]

{'longDescription': {'fr': 'Alors que le soleil est déjà couché, les tout-petits sont invités, en famille, à une rencontre avec le musée, sensorielle, calme, ludique et originale. De quoi faire de jolis rêves !\n\nDurée : 45 min. En famille, enfants de 3 à 5 ans. 1 adulte + 1 enfant obligatoirement.'},
 'participation': [],
 'country': {'br': 'Frañs',
  'de': 'Frankreich',
  'code': 'FR',
  'oc': 'França',
  'en': 'France',
  'it': 'Francia',
  'fr': 'France',
  'es': 'Francia',
  'nl': 'Frankrijk'},
 'featured': False,
 'private': 0,
 'keywords': {'fr': ['visite', 'en famille']},
 'accessibility': {'ii': False,
  'hi': False,
  'vi': False,
  'pi': False,
  'mi': False},
 'dateRange': {'ar': 'الجمعة ٣٠ أكتوبر, 18:00',
  'de': 'Freitag 30 Oktober, 18:00',
  'en': 'Friday 30 October, 18:00',
  'it': 'Venerdì 30 ottobre, 18:00',
  'fr': 'Vendredi 30 octobre, 18h00',
  'es': 'Viernes 30 octubre, 18:00',
  'nl': 'Vrijdag 30 oktober, 18u00'},
 'timezone': 'Europe/Paris',
 'organisateur': []

In [26]:
# ---------- Exemple 2 ----------
events[sample_positions[1]]

{'longDescription': {'fr': 'Cette exposition propose un voyage sensible entre tradition et réinvention, où le fil devient lien de mémoire, de fierté et de création. Une mise en lumière de la richesse du patrimoine textile amazigh à travers deux approches artistiques féminines et engagées : les tapis de l’atelier des tisserandes et les broderies de Nébuleuz.\n\n**Vernissage jeu. 27 nov. - 18h30**'},
 'participation': [42],
 'country': {'br': 'Frañs (Metropol)',
  'de': 'Frankreich (Metropolitan)',
  'code': 'FR',
  'oc': 'França (Metropolitana)',
  'en': 'France (Metropolitan)',
  'it': 'Francia (continente)',
  'fr': 'France (Métropole)',
  'es': 'Francia (Metropolitana)',
  'nl': 'Frankrijk (Europees)'},
 'featured': False,
 'private': 0,
 'keywords': {},
 'accessibility': {'ii': False,
  'hi': False,
  'vi': False,
  'pi': False,
  'mi': False},
 'dateRange': {'ar': '٢٦ نوفمبر ٢٠٢٥ - ١٩ يناير ٢٠٢٦',
  'de': '26 November 2025 - 19 Januar 2026',
  'en': '26 November 2025 - 19 January 2

In [27]:
# ---------- Exemple 3 ----------
events[sample_positions[2]]

{'longDescription': {'fr': "Atelier créatif à partir de 3 ans au Studio Champs Libres du Muséum\n-------------------------------------------------------------------\n\nIl est temps de décorer votre maison ! Nous vous proposons de confectionner des rennes et des sapins.  \n**Cet atelier est accessible aux publics :**\n\n*   non et mal voyant\n*   en situation de handicap mental cognitif et psychique\n*   LSF\n\n### **Date et horaire :**\n\n*   **mardi 23 décembre de 10h30 à 12h30**\n\n### **\\+ d'infos :**\n\n*   tout public à partir de 3 ans\n*   tarif : inclus dans le billet d'entrée, sans inscription\n*   lieu : Studio Champs Libres du Muséum (1er étage)"},
 'participation': [],
 'country': {'br': 'Frañs',
  'de': 'Frankreich',
  'code': 'FR',
  'oc': 'França',
  'en': 'France',
  'it': 'Francia',
  'fr': 'France',
  'es': 'Francia',
  'nl': 'Frankrijk'},
 'featured': False,
 'private': 0,
 'keywords': {'fr': ['lsf',
   'cognitif et psychique',
   'accessibilité',
   'non et mal voya

In [28]:
# ---------- Valeurs du champ types-devenements ----------
type_counts = Counter()

for event in events:
    event_types = event.get("types-devenements")

    if isinstance(event_types, list):
        type_counts.update(event_types)

print(f"Nombre de types différents : {len(type_counts)}")
print("\nTypes rencontrés et nombre d'occurrences :")

for event_type, count in type_counts.most_common():
    print(f"{event_type} : {count}")
    

Nombre de types différents : 12

Types rencontrés et nombre d'occurrences :
24 : 2348
45 : 1718
25 : 1366
20 : 745
16 : 503
22 : 494
44 : 468
19 : 399
15 : 313
21 : 50
17 : 27
23 : 12


Le champ types-devenements contient 12 identifiants différents. En additionnant leurs occurrences, on dépasse le nombre total d'événements collectés : cela montre qu'un même événement peut avoir plusieurs types.

Pour connaître la signification de ces identifiants, nous allons ajouter un appel à l'API OpenAgenda dans openagenda.py. Les types d'événements et leurs libellés seront enregistrés dans un fichier JSON séparé.

Le rapprochement entre les événements et leurs types sera ensuite réalisé pendant le preprocessing afin de remplacer les identifiants numériques par des informations compréhensibles et exploitables par le chatbot.